# 21-06 · Уничтожаем врагов и корабль

Практика к разделам [«Попадания и столкновения»](../../site/chapters/glava-21/21-06-unichtozhenie.html) и [«Завершение игры и экран «Игра окончена»»](../../site/chapters/glava-21/21-07-game-over.html).

## Цель

Проверить, что попадание пули уничтожает врага и начисляет очки, а враг, долетевший до низа экрана, завершает игру.

## Рабочий пример — попадание пули по врагу

In [1]:
import random

import pygame

SHIRINA, VYSOTA = 480, 720
FPS = 60

KORABL_SHIRINA, KORABL_VYSOTA = 44, 44
KORABL_SKOROST = 260.0   # px/s

PULYA_SHIRINA, PULYA_VYSOTA = 6, 18
PULYA_SKOROST = 560.0   # px/s

VRAG_SHIRINA, VRAG_VYSOTA = 32, 28
VRAG_SKOROST = 150.0                # px/s
INTERVAL_POYAVLENIYA_VRAGA = 0.75   # секунд между новыми врагами

BELYJ = (255, 255, 255)
CHERNYJ = (10, 10, 20)
ZELYONYJ = (80, 220, 120)
KRASNYJ = (230, 60, 60)
ZHYOLTYJ = (240, 220, 80)

pygame.init()
screen = pygame.display.set_mode((SHIRINA, VYSOTA))
pygame.display.set_caption("Космический шутер")
clock = pygame.time.Clock()
shrift = pygame.font.SysFont(None, 32)
shrift_bolshoj = pygame.font.SysFont(None, 64)


def novaya_igra():
    korabl = pygame.Rect(
        SHIRINA // 2 - KORABL_SHIRINA // 2,
        VYSOTA - KORABL_VYSOTA - 20,
        KORABL_SHIRINA,
        KORABL_VYSOTA,
    )
    return {
        "korabl": korabl,
        "korabl_x": float(korabl.x),
        "puli": [],
        "vragi": [],
        "schet": 0,
        "vremya_do_vraga": INTERVAL_POYAVLENIYA_VRAGA,
        "igra_okonchena": False,
    }


def obrabotat_klavishi(state, klavishi, dt):
    napravlenie = klavishi[pygame.K_RIGHT] - klavishi[pygame.K_LEFT]
    korabl_x = state["korabl_x"] + napravlenie * KORABL_SKOROST * dt
    state["korabl_x"] = max(0.0, min(korabl_x, SHIRINA - KORABL_SHIRINA))
    state["korabl"].x = round(state["korabl_x"])


def vystrelit(state):
    korabl = state["korabl"]
    pulya_rect = pygame.Rect(
        korabl.centerx - PULYA_SHIRINA // 2,
        korabl.top,
        PULYA_SHIRINA,
        PULYA_VYSOTA,
    )
    state["puli"].append({"rect": pulya_rect, "y": float(pulya_rect.y)})


def sozdat_vraga():
    x = random.randint(0, SHIRINA - VRAG_SHIRINA)
    return {"rect": pygame.Rect(x, -VRAG_VYSOTA, VRAG_SHIRINA, VRAG_VYSOTA), "y": float(-VRAG_VYSOTA)}


def obnovit_igru(state, dt):
    if state["igra_okonchena"]:
        return

    for pulya in state["puli"]:
        pulya["y"] -= PULYA_SKOROST * dt
        pulya["rect"].y = round(pulya["y"])
    state["puli"] = [p for p in state["puli"] if p["rect"].bottom > 0]

    state["vremya_do_vraga"] -= dt
    if state["vremya_do_vraga"] <= 0.0:
        state["vragi"].append(sozdat_vraga())
        state["vremya_do_vraga"] += INTERVAL_POYAVLENIYA_VRAGA

    for vrag in state["vragi"]:
        vrag["y"] += VRAG_SKOROST * dt
        vrag["rect"].y = round(vrag["y"])

    novye_puli = []
    novye_vragi = list(state["vragi"])
    for pulya in state["puli"]:
        popala = False
        for vrag in list(novye_vragi):
            if pulya["rect"].colliderect(vrag["rect"]):
                novye_vragi.remove(vrag)
                state["schet"] += 10
                popala = True
                break
        if not popala:
            novye_puli.append(pulya)
    state["puli"] = novye_puli
    state["vragi"] = novye_vragi

    for vrag in state["vragi"]:
        if vrag["rect"].bottom >= VYSOTA or vrag["rect"].colliderect(state["korabl"]):
            state["igra_okonchena"] = True
            break


def narisovat(state):
    screen.fill(CHERNYJ)
    pygame.draw.rect(screen, ZELYONYJ, state["korabl"])
    for pulya in state["puli"]:
        pygame.draw.rect(screen, ZHYOLTYJ, pulya["rect"])
    for vrag in state["vragi"]:
        pygame.draw.rect(screen, KRASNYJ, vrag["rect"])

    tablo = shrift.render(f"Счёт: {state['schet']}", True, BELYJ)
    screen.blit(tablo, (10, 10))

    if state["igra_okonchena"]:
        nadpis = shrift_bolshoj.render("ИГРА ОКОНЧЕНА", True, BELYJ)
        rect = nadpis.get_rect(center=(SHIRINA // 2, VYSOTA // 2))
        screen.blit(nadpis, rect)

    pygame.display.flip()

pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.6)


In [2]:
state = novaya_igra()
dt = 1 / FPS

# ставим врага прямо перед носом корабля и стреляем
vrag_rect = pygame.Rect(state["korabl"].centerx - 20, state["korabl"].top - 30, VRAG_SHIRINA, VRAG_VYSOTA)
state["vragi"] = [{"rect": vrag_rect, "y": float(vrag_rect.y)}]
vystrelit(state)

for kadr in range(10):
    obnovit_igru(state, dt)
    if state["schet"] > 0:
        break

print("Счёт:", state["schet"])
print("Врагов осталось:", len(state["vragi"]))

Счёт: 10
Врагов осталось: 0


## Проверка результата

In [3]:
assert state["schet"] == 10, "попадание должно добавить 10 очков"
assert len(state["vragi"]) == 0, "уничтоженный враг должен исчезнуть из списка"
print("Верно: враг уничтожен пулей, счёт увеличен на 10.")

Верно: враг уничтожен пулей, счёт увеличен на 10.


## Эксперимент — враг долетает до низа экрана

In [4]:
state2 = novaya_igra()
vrag2_rect = pygame.Rect(100, VYSOTA - VRAG_VYSOTA - 1, VRAG_SHIRINA, VRAG_VYSOTA)
state2["vragi"] = [{"rect": vrag2_rect, "y": float(vrag2_rect.y)}]

obnovit_igru(state2, dt)
narisovat(state2)

print("Игра окончена:", state2["igra_okonchena"])
assert state2["igra_okonchena"] is True
print("Верно: враг, долетевший до низа экрана, завершает игру.")

Игра окончена: True
Верно: враг, долетевший до низа экрана, завершает игру.


## Эксперимент — обновление игры останавливается после конца игры

In [5]:
schet_do = state2["schet"]
vragov_do = len(state2["vragi"])

for kadr in range(20):
    obnovit_igru(state2, dt)

assert state2["schet"] == schet_do
assert len(state2["vragi"]) == vragov_do
print("Верно: после игра окончена состояние больше не меняется.")

Верно: после игра окончена состояние больше не меняется.
